Loading tools

In [1]:
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager", repos = "https://cloud.r-project.org")

BiocManager::install(version = "3.22")

BiocManager::install(c(
    "minfi",
    "IlluminaHumanMethylationEPICmanifest",
    "IlluminaHumanMethylationEPICanno.ilm10b4.hg19"
), update = FALSE, ask = FALSE)

Bioconductor version '3.22' is out-of-date; the current release version '3.23'
  is available with R version '4.6'; see https://bioconductor.org/install

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.r-project.org

Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.3 (2026-03-11)

Old packages: 'DelayedArray', 'Matrix', 'S4Vectors', 'SparseArray', 'XML'

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.r-project.org

Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.3 (2026-03-11)

Warning message:
“package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'minfi' 'IlluminaHumanMethylationEPICmanifest'
  'IlluminaHumanMethylationEPICanno.ilm10b4.hg19'”


In [2]:
library(minfi)
packageVersion("minfi")

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following object is masked from ‘pac

[1] ‘1.56.0’

This fails because the 88 IDAT files aren't all the same physical array size.

In [3]:
library(minfi)

targets <- read.csv('/home/ethan-xiao/food-allergy-biomarkers/data/GSE189148_resting_sample_metadata.csv')

rgSet <- read.metharray.exp(targets = targets)

#Error here: IDAT files have different array sizes

ERROR: Error in read.metharray(basenames = files, extended = extended, verbose = verbose, : [read.metharray] Trying to parse IDAT files with different array size but seemingly all of the same type.
  You can force this by 'force=TRUE', see the man page ?read.metharray


In [6]:
library(illuminaio)

grn_files <- list.files('/home/ethan-xiao/food-allergy-biomarkers/data/GSE189148_idats', pattern = '_Grn.idat.gz$', full.names = TRUE) #Checking different probe counts

probe_counts <- sapply(grn_files, function(f) {
  idat <- readIDAT(f)
  idat$nSNPsRead
})

table(probe_counts)

probe_counts
1051815 1051943 
     23      65 

The two different probe counts (1,051,815 vs. 1,051,943) do not line up with allergy status; all three groups (control, single-food-allergic, multi-food-allergic) contain a mix of both. It's most likely a chip-batch/version artifact. This is why forcing past the error below makes sense.

In [7]:
targets <- read.csv('/home/ethan-xiao/food-allergy-biomarkers/data/GSE189148_resting_sample_metadata.csv')

targets$probe_count <- sapply(paste0(targets$Basename, '_Grn.idat.gz'), function(f) {
  readIDAT(f)$nSNPsRead
})

table(targets$allergy_status, targets$probe_count) #Checking if the split has anything to do with allergic/healthy labels

                      
                       1051815 1051943
  control                    3      12
  multi_food_allergic        2      13
  single_food_allergic       4       9

In [9]:
targets <- read.csv('/home/ethan-xiao/food-allergy-biomarkers/data/GSE189148_resting_sample_metadata.csv')
rgSet <- read.metharray.exp(targets = targets, force = TRUE) #Just forcing through

In [11]:
saveRDS(rgSet, '/home/ethan-xiao/food-allergy-biomarkers/data/rgSet_resting.rds')

- 88 GSE189148 resting-state samples successfully loaded as an RGChannelSet despite an array-size mismatch across IDAT files (1,051,815 vs. 1,051,943 probes), confirmed unrelated to allergy status.
- Saved as rgSet_resting.rds ans and used in notebook 3 (quality control) going forward.